# 🏙 Urban Spatial Econometrics — 표준 모형 비교

> **Columbus (OH, 1980) 데이터**로 공간계량 5대 모형을 실습한다.

**Research Question**: 1980년 Columbus 49개 census tract에서 **범죄율(CRIME)**은
소득(INC)과 주택가치(HOVAL)로 얼마나 설명되며, **공간 자기상관**은 어떻게 다뤄야 하는가?

**Reference**: Anselin, L. (1988). *Spatial Econometrics: Methods and Models*, Ch.12

## 1. 데이터 로드

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from uspatial.data import load_example
from uspatial.weights import build_weights, weights_summary
from uspatial.models import SpatialModel, compare_models, impacts_decomposition
from uspatial.diagnostics import morans_i, local_moran, lm_diagnostics
from uspatial.visualization import (
    plot_choropleth, plot_moran_scatter, plot_lisa_cluster,
    plot_weights_connectivity, plot_residual_map, plot_model_comparison
)

gdf = load_example('columbus')
print(f'Shape: {gdf.shape}')
print(f'CRS: {gdf.crs}')
gdf[['POLYID', 'CRIME', 'INC', 'HOVAL', 'geometry']].head()

## 2. 탐색적 공간 데이터 분석 (ESDA)

### 2.1 변수 분포 지도

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_choropleth(gdf, 'CRIME', cmap='YlOrRd', ax=axes[0], title='범죄율 (CRIME)')
plot_choropleth(gdf, 'INC', cmap='YlGnBu', ax=axes[1], title='소득 (INC)')
plot_choropleth(gdf, 'HOVAL', cmap='Purples', ax=axes[2], title='주택가치 (HOVAL)')
plt.tight_layout()

### 2.2 공간가중행렬 (Queen 인접성)

In [ ]:
w_queen = build_weights(gdf, method='queen')
summary = weights_summary(w_queen)
print(summary)
# {'n': 49, 'mean_neighbors': 4.9, 'min': 2, 'max': 10, 'pct_islands': 0.0, ...}

fig, ax = plt.subplots(figsize=(10, 10))
plot_weights_connectivity(gdf, w_queen, ax=ax)
ax.set_title('Columbus — Queen 인접성 그래프', fontsize=14)

### 2.3 전역 Moran's I

In [ ]:
mi = morans_i(gdf['CRIME'].values, w_queen, permutations=999)
print(f"Moran's I = {mi.statistic:.4f}")
print(f"E[I]      = {mi.expected:.4f}")
print(f"z-score   = {mi.z_score:.3f}")
print(f"p-value   = {mi.p_sim:.4f}")
print(mi.interpret())
# 예상: I ≈ 0.5 (강한 양의 공간자기상관), p < 0.001

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
plot_moran_scatter(gdf['CRIME'].values, w_queen, ax=ax)
ax.set_title(f"Moran Scatter — CRIME (I = {mi.statistic:.3f})", fontsize=14)

### 2.4 LISA 클러스터

In [ ]:
lisa = local_moran(gdf['CRIME'].values, w_queen, significance_level=0.05)

fig, ax = plt.subplots(figsize=(10, 10))
plot_lisa_cluster(gdf, lisa, ax=ax, title='LISA 클러스터 — CRIME')

# 클러스터 구성 확인
print(lisa.cluster_labels.value_counts())

## 3. OLS 베이스라인

In [ ]:
X_vars = ['INC', 'HOVAL']
ols = SpatialModel(gdf, y='CRIME', X=X_vars, method='OLS').fit()
print(ols.summary())

In [ ]:
# 잔차 공간 패턴
fig, ax = plt.subplots(figsize=(10, 10))
plot_residual_map(gdf, ols.residuals, title='OLS Residuals', ax=ax)

# 잔차의 Moran's I → 공간자기상관 잔존 여부
mi_resid = morans_i(ols.residuals, w_queen)
print(f"OLS Residual Moran's I = {mi_resid.statistic:.4f} (p={mi_resid.p_sim:.4f})")

## 4. LM 진단 → 모형 선택

In [ ]:
lm = lm_diagnostics(ols.raw_result, w_queen)
print(f'LM-Lag         = {lm.lm_lag:.3f} (p={lm.lm_lag_p:.4f})')
print(f'LM-Error       = {lm.lm_error:.3f} (p={lm.lm_error_p:.4f})')
print(f'Robust LM-Lag  = {lm.robust_lm_lag:.3f} (p={lm.robust_lm_lag_p:.4f})')
print(f'Robust LM-Err  = {lm.robust_lm_error:.3f} (p={lm.robust_lm_error_p:.4f})')
print(f'→ 추천 모형: {lm.recommendation}')
# Columbus 전형적 결과: LM-Error 더 유의 → SEM 권장

## 5. 4개 모형 추정 및 비교 (OLS / SLM / SEM / SDM)

In [ ]:
slm = SpatialModel(gdf, 'CRIME', X_vars, w_queen, 'SLM').fit()
sem = SpatialModel(gdf, 'CRIME', X_vars, w_queen, 'SEM').fit()
sdm = SpatialModel(gdf, 'CRIME', X_vars, w_queen, 'SDM').fit()

comparison = compare_models([ols, slm, sem, sdm])
comparison

In [ ]:
fig = plot_model_comparison(comparison)
fig.suptitle('모형 적합도 비교: OLS vs SLM vs SEM vs SDM', fontsize=14)

## 6. SLM 해석 — Direct/Indirect/Total Effects

LeSage & Pace (2009): SLM의 β 계수는 **직접 해석 불가**.
총 한계효과 = (I - ρW)⁻¹ X β 로 분해해야 한다.

In [ ]:
impacts = impacts_decomposition(slm)
impacts

## 7. GWR — 계수의 공간적 이질성

In [ ]:
gwr = SpatialModel(gdf, 'CRIME', X_vars, w_queen, 'GWR').fit()

from uspatial.visualization import plot_gwr_coefficients
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_gwr_coefficients(gdf, gwr.raw_result, 'INC', ax=axes[0])
axes[0].set_title('GWR 계수: INC')
plot_gwr_coefficients(gdf, gwr.raw_result, 'HOVAL', ax=axes[1])
axes[1].set_title('GWR 계수: HOVAL')

## 8. 종합 결론

**Columbus 1980 범죄 데이터 분석 결과 요약**:

1. CRIME은 강한 양의 공간자기상관 (Moran's I ≈ 0.5, p < 0.001)
2. OLS 잔차에도 자기상관 잔존 → 모형 오설정
3. LM 진단: LM-Error가 더 유의 → **SEM 채택 권장**
4. SEM의 λ ≈ 0.56 → 관측되지 않은 공간적 공통 요인 존재
5. GWR 결과: INC의 한계효과가 북부 저소득 지역에서 더 강하게 음(-)

**정책적 시사점**: 단순 OLS 회귀로 범죄율-소득 관계를 추정하면 **공간 이질성을 놓침**.
국지적 개입(지역 맞춤형 정책) 설계 시 GWR의 국지 계수를 참고해야 한다.